In [1]:
!pip install pandas scikit-learn xgboost lightgbm joblib


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import joblib

# Load your dataset
df = pd.read_csv("/content/ev_battery_soh_dataset.csv")  # Adjust path as needed

# Features and Target
X = df.drop('SOH (%)', axis=1)
y = df['SOH (%)']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
models = {
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train, Evaluate, Save Models
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Evaluation
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"{name} Results:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"R2 Score: {r2:.4f}")

    # Save model
    joblib.dump(model, f"{name}_SOH_model.pkl")
    print(f"{name} model saved as {name}_SOH_model.pkl")



Training XGBoost...
XGBoost Results:
MAE: 1.1937
MSE: 2.2137
R2 Score: 0.9601
XGBoost model saved as XGBoost_SOH_model.pkl

Training LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003526 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 8
[LightGBM] [Info] Start training from score 85.554276
LightGBM Results:
MAE: 1.1897
MSE: 2.2029
R2 Score: 0.9603
LightGBM model saved as LightGBM_SOH_model.pkl

Training RandomForest...
RandomForest Results:
MAE: 1.2154
MSE: 2.3096
R2 Score: 0.9583
RandomForest model saved as RandomForest_SOH_model.pkl


In [8]:
import joblib
import numpy as np

# Load the saved model (XGBoost example here)
model = joblib.load('/content/XGBoost_SOH_model.pkl')

# Prediction function
def predict_soh(voltage, current, temperature, charge_cycles,
                discharge_cycles, avg_charge_rate, avg_discharge_rate, time_elapsed):
    """
    Predicts SOH (%) of an EV battery using 8 input features.
    """
    features = np.array([[voltage, current, temperature, charge_cycles,
                          discharge_cycles, avg_charge_rate, avg_discharge_rate, time_elapsed]])

    soh_prediction = model.predict(features)
    return soh_prediction[0]

# Example input values
voltage = 3.88564
current = 1.506667
temperature = 32.05252
charge_cycles = 1392
discharge_cycles = 1385
avg_charge_rate = 1.303963
avg_discharge_rate = 1.507909
time_elapsed = 2186.142

# Predict SOH
predicted_soh = predict_soh(voltage, current, temperature, charge_cycles,
                             discharge_cycles, avg_charge_rate, avg_discharge_rate, time_elapsed)

print(f"🔋 Predicted SOH: {predicted_soh:.2f}%")


🔋 Predicted SOH: 74.95%
